# Notebook 03 — Grad-CAM Visualisation

This notebook loads the trained CNN and applies our **from-scratch Grad-CAM** implementation
to understand *what spatial regions* the model attends to when classifying each retinal condition.

## What should the model be looking at?

| Class  | Expected region of attention                                               |
|--------|----------------------------------------------------------------------------|
| CNV    | The subretinal membrane below the RPE; fluid pockets beneath the retina    |
| DME    | Intraretinal hyporeflective (dark) pockets within the retinal layers       |
| DRUSEN | The bumpy, irregular RPE surface; yellow sub-RPE deposits                  |
| NORMAL | The smooth foveal pit contour and evenly layered RPE / IS-OS band          |

If the heatmap highlights anatomically correct regions, we have evidence the model
has internalized clinically meaningful features — not shortcuts (e.g. image artifacts).

In [ ]:
import sys, os
sys.path.insert(0, '..')
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2

from src.model_cnn import RetinalCNN
from src.dataset   import get_dataloaders
from src.gradcam   import GradCAM, _denormalize

plt.rcParams['figure.dpi'] = 110
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Load model ──────────────────────────────────────────────────────────────
model = RetinalCNN(num_classes=4).to(DEVICE)
ckpt  = torch.load(f'{RESULTS_DIR}/best_cnn.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f"Loaded CNN checkpoint — epoch {ckpt['epoch']}, val_acc={ckpt['val_acc']*100:.2f}%")

# ── Load data ───────────────────────────────────────────────────────────────
_, _, test_loader = get_dataloaders('../data', batch_size=32, num_workers=4)

# Collect 3 images per class from the test set
N_PER_CLASS = 3
class_images = {c: [] for c in CLASS_NAMES}

for images, labels in test_loader:
    for img, lbl in zip(images, labels):
        cls = CLASS_NAMES[lbl.item()]
        if len(class_images[cls]) < N_PER_CLASS:
            class_images[cls].append(img)
    if all(len(v) == N_PER_CLASS for v in class_images.values()):
        break

print(f"Collected {N_PER_CLASS} images per class for Grad-CAM.")

## Grad-CAM on 3 Images × 4 Classes = 12 Total

Layout: **Row 0 = original · Row 1 = Grad-CAM overlay**  
Each column pair corresponds to one image.

The heatmap is a weighted sum of the last convolutional layer's feature maps,
weighted by the gradient of the target class score w.r.t. each feature map.
High-intensity (red/yellow) regions are *most responsible* for the prediction.

In [ ]:
target_layer = model.get_last_conv_layer()
gcam         = GradCAM(model, target_layer)

fig, axes = plt.subplots(2 * 4, N_PER_CLASS, figsize=(N_PER_CLASS * 4, 2 * 4 * 3))
fig.suptitle('Grad-CAM — RetinalCNN  (top: original · bottom: heatmap overlay)',
             fontsize=14, fontweight='bold', y=1.01)

class_descriptions = {
    'CNV':    'Expected: subretinal fluid / membrane below RPE',
    'DME':    'Expected: intraretinal hyporeflective pockets',
    'DRUSEN': 'Expected: bumpy irregular RPE surface',
    'NORMAL': 'Expected: smooth foveal pit, orderly layers',
}

for cls_idx, cls in enumerate(CLASS_NAMES):
    for col, img_tensor in enumerate(class_images[cls]):
        inp = img_tensor.unsqueeze(0).to(DEVICE)
        heatmap = gcam.generate(inp)             # [H, W] in [0,1]
        raw_img = _denormalize(img_tensor)        # uint8 [H, W, 3]
        overlay = gcam.overlay_heatmap(heatmap, raw_img.copy())

        row_orig  = 2 * cls_idx
        row_cam   = 2 * cls_idx + 1

        axes[row_orig, col].imshow(raw_img)
        axes[row_orig, col].axis('off')
        if col == 0:
            axes[row_orig, col].set_ylabel(
                f"{cls}\n{class_descriptions[cls]}",
                fontsize=9, fontweight='bold', rotation=0, ha='right', labelpad=5
            )

        axes[row_cam, col].imshow(overlay)
        axes[row_cam, col].axis('off')
        if col == 0:
            axes[row_cam, col].set_ylabel('Grad-CAM', fontsize=9, rotation=0, ha='right')

gcam.remove_hooks()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/gradcam_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {RESULTS_DIR}/gradcam_grid.png")

## Clinical Interpretation

Run the cell above and visually inspect each row:

- **CNV**: Does the heatmap highlight the region *below* the RPE (bottom third of the scan)? Subretinal fluid in CNV causes a dome-shaped bulge beneath the RPE line.  
- **DME**: Heatmap should concentrate on the middle retinal layers where intraretinal cysts (dark round pockets) appear.  
- **DRUSEN**: Expect attention along the wavy RPE surface — drusen appear as bumps that elevate the RPE irregularly. If the model fires on the center of the image instead, it may be using a positional shortcut.  
- **NORMAL**: Attention should be diffuse or focus on the foveal pit. A sharp heatmap on a featureless region of a normal scan may indicate uncertainty.  

Misaligned heatmaps (e.g. model firing on image borders or artifacts) are a red flag that the model has learned dataset-specific biases rather than true pathology.

In [ ]:
# 2×4 grid (1 image per class) — this is the version saved as gradcam_grid.png
from src.gradcam import generate_gradcam_grid

one_per_class = [class_images[c][0].unsqueeze(0) for c in CLASS_NAMES]
gcam2 = GradCAM(model, model.get_last_conv_layer())

generate_gradcam_grid(
    model        = model,
    target_layer = model.get_last_conv_layer(),
    images       = one_per_class,
    class_names  = CLASS_NAMES,
    save_path    = f'{RESULTS_DIR}/gradcam_grid.png',
    device       = str(DEVICE),
)
gcam2.remove_hooks()